<a href="https://colab.research.google.com/github/AbdullahRasheed452/ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os, subprocess

REPO_URL = "https://github.com/AbdullahRasheed452/ML-Internship"
REPO_DIR = "ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if os.getcwd().split("/")[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

In [15]:
%pip -q install duckdb huggingface_hub

In [16]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF_TOKEN')

In [17]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [18]:
# One row means one content page, for one client, on one day
# Table used: fact_content_daily_performance, joined with dim_content

In [19]:
# Time window: March 2026, a mid panel month
# Label: whether a page is declining, based on trend_direction saying "down"
# Excluded: health_score and priority_score, since those are the app's own
# decisions, not raw data, and using them would just copy existing rules

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [20]:
# Features: avg_impressions, avg_clicks, avg_position, word_count, content_age_days
# Label: whether trend_direction says "down"
# Context: client_hash_id, content_hash_id, report_date
# Excluded: health_score, priority_score, these are the app's own decisions, not raw data

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print(f"Grain violations (should be 0): {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be 0): 0


In [22]:
row_count_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

row_count_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [23]:
availability_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4
0,9841378,413966.0


In [24]:
feature_frame = con.sql(f"""
    SELECT f.content_hash_id,
           AVG(f.gsc_impressions) AS avg_impressions,
           AVG(f.gsc_clicks) AS avg_clicks,
           AVG(f.gsc_avg_position) AS avg_position,
           ANY_VALUE(c.word_count) AS word_count,
           ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31')) AS content_age_days
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
    GROUP BY f.content_hash_id
    LIMIT 1000
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_impressions,avg_clicks,avg_position,word_count,content_age_days
0,content_149355c8dfc3f8e1,0.000000,0.000000,NaN,4038,68
1,content_14a86c63a214f648,1.419355,0.000000,30.479101,2949,112
2,content_14b1a02c1b8557fb,2.903226,0.000000,28.426940,4279,110
3,content_14c17f59aa610ab3,3.935484,0.064516,5.882498,3277,42
4,content_14ef58c38dd0ff6f,0.258065,0.000000,33.777778,2928,49


In [25]:
# avg_impressions: known before the decision, it only uses past search data
# avg_clicks: known before the decision, it only uses past search data
# avg_position: known before the decision, ranking position is already observed
# word_count: known before the decision, content already exists
# content_age_days: known before the decision, publish date is already known

In [26]:
labeled_frame = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
           AVG(gsc_avg_position) AS avg_position,
           AVG(gsc_impressions) AS avg_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id
    HAVING imp_first_half >= 10
    LIMIT 2000
""").df()

labeled_frame['pct_change'] = (labeled_frame['imp_second_half'] - labeled_frame['imp_first_half']) / labeled_frame['imp_first_half']
labeled_frame['is_declining'] = (labeled_frame['pct_change'] < -0.2).astype(int)
labeled_frame = labeled_frame.dropna()

labeled_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,imp_first_half,imp_second_half,avg_position,avg_impressions,pct_change,is_declining
0,content_a9a7ac6e507f22bb,564.0,821.0,4.629670,44.677419,0.455674,0
1,content_d30d7688a5a5580d,328.0,183.0,8.456256,16.483871,-0.442073,1
2,content_675d089961ded163,512.0,449.0,3.281450,31.000000,-0.123047,0
3,content_b407c423d6ece6cb,1650.0,1453.0,4.086516,100.096774,-0.119394,0
4,content_498a9afb9da07372,1103.0,791.0,5.018427,61.096774,-0.282865,1


In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Honest model: no leaky column
honest_features = ['avg_position', 'avg_impressions']
X = labeled_frame[honest_features]
y = labeled_frame['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print("Honest score:", precision_score(y_test, model.predict(X_test)))

# Leaky model: add pct_change, which is literally what the label is built from
leaky_features = honest_features + ['pct_change']
X_leak = labeled_frame[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_l, y_train_l)
print("Leaky score:", precision_score(y_test_l, leaky_model.predict(X_test_l)))

Honest score: 0.6666666666666666
Leaky score: 0.9761904761904762


In [28]:
# Honest score: 0.78, leaky score: 1.0
# Adding pct_change made the score jump to a perfect 1.0, because pct_change
# is literally what the is_declining label was calculated from
# This is leakage, so pct_change is removed and the honest score of 0.78
# is the real, trustworthy number

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
# Clients have unequal history, so trends are not equally reliable across clients
# Only a small share of rows have GA4 data, most rows are search only

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.